# Daily Audit Summary Report

**Purpose:** Generate comprehensive daily operational visibility and compliance summary.

**Data Sources:**
* `workspace.audit.audit_pipeline_runs` - Pipeline execution metrics
* `workspace.audit.audit_dq_results` - Data quality validation results
* `workspace.audit.audit_access_events` - Access audit trail

**Report Sections:**
1. Pipeline Execution Summary (success rate, failures, performance)
2. Data Quality Health (rule violations by severity)
3. Security & Access Patterns (access attempts, denials)
4. SLA Compliance Metrics
5. Critical Alerts Summary

**Schedule:** Run daily at 6:00 AM to summarize previous day's operations

In [0]:
# Notebook parameters
dbutils.widgets.text("report_date", "", "Report Date (YYYY-MM-DD, default=yesterday)")
dbutils.widgets.text("environment", "PROD", "Environment Filter (ALL/DEV/TEST/PROD)")

# Get parameter values
from datetime import datetime, timedelta

report_date_str = dbutils.widgets.get("report_date")
if report_date_str:
    report_date = datetime.strptime(report_date_str, "%Y-%m-%d").date()
else:
    report_date = (datetime.now() - timedelta(days=1)).date()

environment_filter = dbutils.widgets.get("environment")

print(f"Generating Daily Audit Summary Report")
print(f"Report Date: {report_date}")
print(f"Environment: {environment_filter}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import json

In [0]:
%sql
-- Pipeline execution metrics for the report date
-- Aggregates pipeline runs, success rates, and performance

CREATE OR REPLACE TEMP VIEW pipeline_summary AS
SELECT 
    DATE(start_time) as report_date,
    environment,
    COUNT(*) as total_runs,
    SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) as successful_runs,
    SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) as failed_runs,
    SUM(CASE WHEN status = 'RUNNING' THEN 1 ELSE 0 END) as running_runs,
    ROUND(SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as success_rate_pct,
    SUM(rows_read) as total_rows_read,
    SUM(rows_written) as total_rows_written,
    ROUND(AVG(runtime_seconds), 2) as avg_runtime_seconds,
    ROUND(MAX(runtime_seconds), 2) as max_runtime_seconds,
    COUNT(DISTINCT pipeline_name) as unique_pipelines
FROM workspace.audit.audit_pipeline_runs
WHERE DATE(start_time) = CURRENT_DATE() - INTERVAL 1 DAY
GROUP BY DATE(start_time), environment;

SELECT * FROM pipeline_summary
ORDER BY environment;

In [0]:
%sql
-- Data quality validation results summary
-- Groups by severity and identifies critical quality issues

CREATE OR REPLACE TEMP VIEW dq_summary AS
SELECT 
    DATE(evaluated_at) as report_date,
    severity,
    COUNT(*) as total_checks,
    SUM(CASE WHEN failed_records > 0 THEN 1 ELSE 0 END) as failed_checks,
    SUM(failed_records) as total_failed_records,
    COUNT(DISTINCT CONCAT(target_schema, '.', target_table)) as affected_tables,
    COLLECT_LIST(
        CASE WHEN failed_records > 0 
        THEN CONCAT(rule_name, ' (', target_schema, '.', target_table, '): ', failed_records, ' failures')
        END
    ) as failure_details
FROM workspace.audit.audit_dq_results
WHERE DATE(evaluated_at) = CURRENT_DATE() - INTERVAL 1 DAY
GROUP BY DATE(evaluated_at), severity;

SELECT 
    severity,
    total_checks,
    failed_checks,
    total_failed_records,
    affected_tables,
    CASE 
        WHEN severity = 'ERROR' AND failed_checks > 0 THEN '⚠️ CRITICAL ATTENTION REQUIRED'
        WHEN severity = 'WARNING' AND failed_checks > 0 THEN '⚠️ Review Recommended'
        ELSE '✓ All checks passed'
    END as status_flag
FROM dq_summary
ORDER BY 
    CASE severity 
        WHEN 'ERROR' THEN 1 
        WHEN 'WARNING' THEN 2 
        WHEN 'INFO' THEN 3 
    END;

In [0]:
%sql
-- Access audit summary
-- Tracks access patterns, denied attempts, and security events

CREATE OR REPLACE TEMP VIEW access_summary AS
SELECT 
    DATE(event_time) as report_date,
    COUNT(*) as total_access_events,
    COUNT(DISTINCT actor) as unique_actors,
    COUNT(DISTINCT object_name) as unique_objects_accessed,
    SUM(CASE WHEN result = 'SUCCESS' THEN 1 ELSE 0 END) as successful_accesses,
    SUM(CASE WHEN result = 'DENIED' THEN 1 ELSE 0 END) as denied_accesses,
    SUM(CASE WHEN action_type IN ('WRITE', 'DELETE') THEN 1 ELSE 0 END) as write_operations,
    SUM(CASE WHEN action_type = 'READ' THEN 1 ELSE 0 END) as read_operations
FROM workspace.audit.audit_access_events
WHERE DATE(event_time) = CURRENT_DATE() - INTERVAL 1 DAY
GROUP BY DATE(event_time);

SELECT 
    *,
    CASE 
        WHEN denied_accesses > 0 THEN CONCAT('⚠️ ', denied_accesses, ' denied access attempts detected')
        ELSE '✓ No security incidents'
    END as security_status
FROM access_summary;

In [0]:
%sql
-- Identify pipelines requiring immediate attention
-- Shows pipelines with failures in the last 24 hours

SELECT 
    pipeline_name,
    environment,
    COUNT(*) as total_runs,
    SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) as failures,
    ROUND(SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as failure_rate_pct,
    MAX(start_time) as last_run,
    FIRST(error_message) as sample_error
FROM workspace.audit.audit_pipeline_runs
WHERE start_time >= CURRENT_TIMESTAMP() - INTERVAL 1 DAY
  AND status = 'FAILED'
GROUP BY pipeline_name, environment
HAVING failures > 0
ORDER BY failures DESC, failure_rate_pct DESC
LIMIT 10;

In [0]:
%sql
-- Data quality rules with ERROR severity failures
-- Requires immediate remediation

SELECT 
    rule_name,
    CONCAT(target_schema, '.', target_table) as target_table,
    COUNT(*) as failure_occurrences,
    SUM(failed_records) as total_failed_records,
    MAX(evaluated_at) as last_failure_time,
    DATEDIFF(HOUR, MAX(evaluated_at), CURRENT_TIMESTAMP()) as hours_since_last_failure
FROM workspace.audit.audit_dq_results
WHERE evaluated_at >= CURRENT_TIMESTAMP() - INTERVAL 1 DAY
  AND severity = 'ERROR'
  AND failed_records > 0
GROUP BY rule_name, target_schema, target_table
ORDER BY total_failed_records DESC, failure_occurrences DESC
LIMIT 15;

In [0]:
%sql
-- Track denied access attempts for security review
-- Identifies potential unauthorized access attempts

SELECT 
    actor,
    action_type,
    object_name,
    object_type,
    COUNT(*) as denied_attempts,
    MIN(event_time) as first_attempt,
    MAX(event_time) as last_attempt,
    CASE 
        WHEN COUNT(*) > 5 THEN '⚠️ HIGH - Multiple attempts'
        WHEN COUNT(*) > 2 THEN '⚠️ MEDIUM - Several attempts'
        ELSE 'LOW - Single/Few attempts'
    END as threat_level
FROM workspace.audit.audit_access_events
WHERE event_time >= CURRENT_TIMESTAMP() - INTERVAL 1 DAY
  AND result = 'DENIED'
GROUP BY actor, action_type, object_name, object_type
ORDER BY denied_attempts DESC, last_attempt DESC
LIMIT 20;

In [0]:
# Calculate SLA compliance metrics
# Define SLA thresholds
SLA_SUCCESS_RATE_THRESHOLD = 95.0  # 95% success rate
SLA_MAX_RUNTIME_THRESHOLD = 3600   # 1 hour max runtime
SLA_DQ_ERROR_THRESHOLD = 0         # Zero ERROR-level DQ failures
SLA_SECURITY_DENIAL_THRESHOLD = 5  # Max 5 denied access attempts

# Get summary data
pipeline_summary_df = spark.sql("SELECT * FROM pipeline_summary")
dq_summary_df = spark.sql("SELECT * FROM dq_summary")
access_summary_df = spark.sql("SELECT * FROM access_summary")

# Calculate compliance
pipeline_row = pipeline_summary_df.filter(F.col("environment") == "PROD").first()
if pipeline_row:
    success_rate = pipeline_row["success_rate_pct"]
    max_runtime = pipeline_row["max_runtime_seconds"]
    pipeline_sla_met = success_rate >= SLA_SUCCESS_RATE_THRESHOLD and max_runtime <= SLA_MAX_RUNTIME_THRESHOLD
else:
    pipeline_sla_met = True  # No runs = no SLA breach
    success_rate = None
    max_runtime = None

dq_errors = dq_summary_df.filter(F.col("severity") == "ERROR").agg(F.sum("failed_checks")).first()[0] or 0
dq_sla_met = dq_errors <= SLA_DQ_ERROR_THRESHOLD

access_row = access_summary_df.first()
if access_row:
    denied_accesses = access_row["denied_accesses"]
    security_sla_met = denied_accesses <= SLA_SECURITY_DENIAL_THRESHOLD
else:
    security_sla_met = True
    denied_accesses = 0

overall_sla_met = pipeline_sla_met and dq_sla_met and security_sla_met

print("="*60)
print("SLA COMPLIANCE SUMMARY")
print("="*60)
print(f"Pipeline SLA: {'✓ MET' if pipeline_sla_met else '✗ BREACH'}")
print(f"  - Success Rate: {success_rate}% (target: >={SLA_SUCCESS_RATE_THRESHOLD}%)")
print(f"  - Max Runtime: {max_runtime}s (target: <={SLA_MAX_RUNTIME_THRESHOLD}s)")
print(f"\nData Quality SLA: {'✓ MET' if dq_sla_met else '✗ BREACH'}")
print(f"  - ERROR-level failures: {dq_errors} (target: <={SLA_DQ_ERROR_THRESHOLD})")
print(f"\nSecurity SLA: {'✓ MET' if security_sla_met else '✗ BREACH'}")
print(f"  - Denied accesses: {denied_accesses} (target: <={SLA_SECURITY_DENIAL_THRESHOLD})")
print(f"\nOVERALL SLA STATUS: {'✓ MET' if overall_sla_met else '✗ BREACH'}")
print("="*60)

In [0]:
# Build structured summary report for downstream consumption
summary_report = {
    "report_date": str(report_date),
    "generated_at": datetime.now().isoformat(),
    "environment": environment_filter,
    "sla_compliance": {
        "overall_met": overall_sla_met,
        "pipeline_met": pipeline_sla_met,
        "data_quality_met": dq_sla_met,
        "security_met": security_sla_met
    },
    "metrics": {
        "pipeline_success_rate": float(success_rate) if success_rate else 0.0,
        "dq_error_failures": dq_errors,
        "security_denied_accesses": denied_accesses
    },
    "requires_attention": not overall_sla_met
}

# Store report as JSON string for notification dispatch
report_json = json.dumps(summary_report, indent=2)
print("\nGenerated summary report:")
print(report_json)

# Store report in widget for downstream notebooks
dbutils.jobs.taskValues.set(key="daily_summary_report", value=report_json)

In [0]:
# Return summary report for orchestration
dbutils.notebook.exit(summary_report)